<a href="https://colab.research.google.com/github/briskicedteaa/Filler-Name/blob/main/FOXP2_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install BioPython
from Bio import SeqIO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.8 MB/s eta 0:00:00


In [ ]:
import gzip

with gzip.open("uniprotkb_FOXP2_2026_08_15.fasta.gz", "rt") as handle:
  records = list(SeqIO.parse(handle, "fasta"))

In [ ]:
print(records[0])
print(records[1])

ID: sp|O00560|SDCB1_HUMAN
Name: sp|O00560|SDCB1_HUMAN
Description: sp|O00560|SDCB1_HUMAN Syntenin-1 OS=Homo sapiens OX=9606 GN=SDCBP PE=1 SV=1
Number of features: 0
Seq('MSLYPSLEDLKVDKVIQAQTAFSANPANPAILSEASAPIPHDGNLYPRLYPELS...PEV')
ID: sp|O15409|FOXP2_HUMAN
Name: sp|O15409|FOXP2_HUMAN
Description: sp|O15409|FOXP2_HUMAN Forkhead box protein P2 OS=Homo sapiens OX=9606 GN=FOXP2 PE=1 SV=2
Number of features: 0
Seq('MMQESATETISNSSMNQNGMSTLSSQLDAGSRDGRSSGDTSSEVSTVELLHLQQ...DLE')


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "ID": [record.id for record in records],
    "Description": [record.description for record in records],
    "Sequence": [str(record.seq) for record in records],
    "Length": [len(record.seq) for record in records],
})

foxp2_df = df[
    df["Description"].str.contains(
        "Forkhead box protein P2",
        case=False,
        na=False
    )
].copy()

foxp2_df = foxp2_df[
    ~foxp2_df["Description"].str.contains(
        "Fragment",
        case=False,
        na=False
    )
].copy()

foxp2_df = foxp2_df.drop_duplicates(
    subset="Sequence",
    keep="first"

).copy()

print("Remaining Proteins:", len(foxp2_df))
print("Total Proteins:", len(df))
print("Duplicate sequences:", len(foxp2_df) - foxp2_df["Sequence"].nunique())
df.head()

Remaining Proteins: 525
Total Proteins: 1227
Duplicate sequences: 0


,ID,Description,Sequence,Length
0,sp|O00560|SDCB1_HUMAN,sp|O00560|SDCB1_HUMAN Syntenin-1 OS=Homo sapie...,MSLYPSLEDLKVDKVIQAQTAFSANPANPAILSEASAPIPHDGNLY...,298
1,sp|O15409|FOXP2_HUMAN,sp|O15409|FOXP2_HUMAN Forkhead box protein P2 ...,MMQESATETISNSSMNQNGMSTLSSQLDAGSRDGRSSGDTSSEVST...,715
2,sp|O88712|CTBP1_MOUSE,sp|O88712|CTBP1_MOUSE C-terminal-binding prote...,MGSSHLLNKGLPLGVRPPIMNGPMHPRPLVALLDGRDCTVEMPILK...,441
3,sp|P0CF24|FOXP2_RAT,sp|P0CF24|FOXP2_RAT Forkhead box protein P2 OS...,MMQESATETISNSSMNQNGMSTLSSQLDAGSRDGRSSGDTSSEVST...,710
4,sp|P24863|CCNC_HUMAN,sp|P24863|CCNC_HUMAN Cyclin-C OS=Homo sapiens ...,MAGNFWQSSHYLQWILDKQDLLKERQKDLKFLSEEEYWKLQIFFTN...,283


In [ ]:
from google.colab import files

foxp2_df.to_csv('FOXP2-Proteins.csv', index = False)
files.download('FOXP2-Proteins.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
with open("FOXP2.fasta", "w") as f:
    for _, row in foxp2_df.iterrows():
        f.write(f">{row['ID']}\n")
        f.write(f"{row['Sequence']}\n")

files.download("FOXP2.fasta")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
foxp2_df[
    ["ID", "Description", "Length"]
].sort_values("Length").head(30)

foxp2_df["Length"].describe()

foxp2_df = foxp2_df[
    foxp2_df["Length"] >= 570
].copy()

print("Proteins remaining:", len(foxp2_df))
print("Below 570 aa:", (foxp2_df["Length"] < 570).sum())
print("570 aa or longer:", (foxp2_df["Length"] >= 570).sum())

Proteins remaining: 494
Below 570 aa: 0
570 aa or longer: 494


In [ ]:
foxp2_df.to_csv("FinalFOXP2.csv", index=False)
files.download("FinalFOXP2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
with open("FOXP2.fasta", "w") as f:
    for _, row in foxp2_df.iterrows():
        f.write(f">{row['ID']}\n")
        f.write(f"{row['Sequence']}\n")

files.download("FOXP2.fasta")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
from Bio import AlignIO

alignment = AlignIO.read(
    "clustalo-I20260815-200247-0706-9045984-p1m.aln-clustal_num",
    "clustal"
)

human = next(record for record in alignment if "O15409" in record.id)
mutations = []
human_position = 0

for i, human_aa in enumerate(human.seq):

  if human_aa != "-":
    human_position += 1

  for record in alignment:

    if record.id == human.id:
      continue

    other_aa = record.seq[i]

    if human_aa == other_aa:
      continue

    if human_aa == "-":
      position = None
    else:
      position = human_position

    mutations.append({
        "ID": record.id,
        "Aligned Position": i + 1,
        "Human Position": position,
        "Human AA": human_aa,
        "Other AA": other_aa,
        "Gap": human_aa == "-" or other_aa == "-"
    })

mutation_df = pd.DataFrame(mutations)

print("Substitutions:", (~mutation_df["Gap"]).sum())
print("Gaps/indels:", mutation_df["Gap"].sum())
print("Total Mutation Rows:", len(mutation_df))
mutation_df.head()

Substitutions: 11157
Gaps/indels: 31026
Total Mutation Rows: 42183


,ID,Aligned Position,Human Position,Human AA,Other AA,Gap
0,tr|A0ACF8DDW5|A0ACF8DDW5_ARAGI,1,NaN,-,M,True
1,tr|A0ACF8DDW5|A0ACF8DDW5_ARAGI,2,NaN,-,S,True
2,tr|A0ACF8DDW5|A0ACF8DDW5_ARAGI,3,NaN,-,V,True
3,tr|A0ACF8DDW5|A0ACF8DDW5_ARAGI,4,NaN,-,Q,True
4,tr|A0A6P7NA03|A0A6P7NA03_BETSP,5,NaN,-,M,True


In [40]:
substitution_df = mutation_df[
    ~mutation_df["Gap"]
].copy()

indel_df = mutation_df[
    mutation_df["Gap"]
].copy()

substitution_df["Mutation"] = (
    substitution_df["Human AA"].astype(str)
    + substitution_df["Human Position"].astype(int).astype(str)
    + substitution_df["Other AA"].astype(str)
)

print("Substitutions:", len(substitution_df))
print("Gap-containing differences:", len(indel_df))
print(
    "Unique substitutions:",
    substitution_df["Mutation"].nunique()
)

substitution_df["Mutation"].value_counts().head(20)

Substitutions: 11157
Gap-containing differences: 31026
Unique substitutions: 1781


,count
Mutation,
N303T,441
S325N,371
S235N,167
A249S,126
S78G,90
Q189P,89
Q191P,89
Q188P,83
S42T,81


In [46]:
mutation_df.to_csv("FOXP2_mutations.csv", index=False)
substitution_df.to_csv("FOXP2_substitutions.csv", index=False)
indel_df.to_csv("FOXP2_indels.csv", index=False)

files.download("FOXP2_mutations.csv")
files.download("FOXP2_substitutions.csv")
files.download("FOXP2_indels.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
unique_substitutions = (
    substitution_df[
        ["Mutation", "Human Position", "Human AA", "Other AA"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

unique_substitutions.to_csv(
    "FOXP2_unique_substitutions.csv",
    index=False
)

print("Unique substitutions:", len(unique_substitutions))

Unique substitutions: 1781
